# Predictive maintenance workshop

#### ✅ How to use this notebook

1. To save your own progress: Go to **File → Save a copy in Drive** (recommended)
2. The cell below will automatically **download all required data** from the course repository.  
3. You can run cells one by one using the play button on the top left of the cell or `Shift + Enter`.

**⚠️ Note: All work done here is temporary unless you save your own copy. Each time you reopen the notebook, Colab starts a new session.**

In [ ]:
# =========================
# 1. Install uv
# =========================
import shutil, sys, os, subprocess

if shutil.which("uv") is None:
    !curl -LsSf https://astral.sh/uv/install.sh | sh
else:
    print(f"uv already installed at {shutil.which('uv')}")

os.environ["PATH"] += ":/root/.local/bin"

# =========================
# 2. Clone repo
# =========================
REPO_URL = "https://github.com/bbrisch/summer_school_prog_dm.git"
REPO_DIR = "summer_school_prog_dm"
REPO_PATH = f"/content/{REPO_DIR}"

if 'google.colab' in sys.modules:
    if os.path.exists(REPO_PATH):
        %cd $REPO_PATH
    else:
        %cd /content
        !git clone $REPO_URL
        %cd $REPO_PATH
else:
    print("Not running in Google Colab environment.")


# =========================
# 3. Create isolated venv if needed
# =========================
if not os.path.exists(".venv"):
    print("Creating uv virtual environment...")
    !uv venv .venv

else:
    print("Using existing .venv")

# =========================
# 4. Sync dependencies INSIDE venv using uv.lock
# =========================
!uv sync --python .venv/bin/python

# =========================
# 5. Ensure notebook uses venv Python (IMPORTANT)
# =========================
venv_site_packages = subprocess.check_output(
    [
        ".venv/bin/python",
        "-c",
        "import site; print(site.getsitepackages()[0])"
    ],
    text=True,
).strip()

sys.path.insert(0, venv_site_packages)
#sys.path.insert(0, f"/content/{REPO_DIR}/.venv/lib/python3.12/site-packages")

print("✅ uv environment ready")

In [ ]:
# %load_ext autoreload
# %autoreload 2

# Import the necessary libraries 
import numpy as np
from numpy import ndarray
from typing import Any

import matplotlib.pyplot as plt
%matplotlib inline

from tqdm import tqdm

from src_prognostics.utils import load_data, format_data, save_prognostics
from src_prognostics.hsmm import CustomHSMM, predict_hsmm_pdf_staked, predict_hsmm_bounds
from src_prognostics.metrics import mae, picp, pinaw, crps
from src_prognostics.plots import plot_rul_bounds, plot_rul_bounds_multiple

from src_decision_making.decision_config import config
from src_decision_making.replacement_agent import ReplacementAgent
import src_decision_making.visualization as vis

# Assignment 📓

## Setup and dataset description


You receive a dataset from run-to-failure experiments on structures. The dataset contains a series of measurements of stiffness reduction that can serve as a health indicator. For this case, the end-of-life event occurs when the stiffness reduction reaches 30%. In addition, the experiments were conducted under two distinct predefined loading conditions. 

Using this dataset, train a hidden semi-Markov model (HSMM) for remaining useful life (RUL) prognostics. Then, you will have to enhance the reliability of the obtained predictions by implementing an uncertainty management (UM) strategy.

To avoid overfitting your results, the data was split into three subsets:
- The training subset will be used to fit model parameters
- The validation subset will be used to evaluate the performance of the model and tune the UM strategy.
- The testing subset will be used to assess the final performance of the model.


The workshop is divided in two sections. The prognostics section consists on training a prognostic model and managing its uncertainty. Then, the desicion-maling section consists on creating a predictive maintenance policy, using the trained prognostic model.

# Section 1 - Prognostics & Uncertainty Management 🔍 📊

## Tasks description

This workshop considers the following tasks:
1. Visualize the provided data, explore its contents, and gather some insights
2. Use the provided codes to train a baseline HSMM. Reflect on the achieved accuracy and the representation of uncertainty.
3. Reflect on how you incorporate uncertainty management in the HSMM.
4. Implement your uncertainty management strategy and analyze its effectiveness
5. Compute all prognostic distributions for all training, validation, and testing datasets. These results will be later used for decision-making.

The following cells contain code examples you can use, as well as some functionalities that are already implemented.in the source code. Feel free to use them. 
In case you have questions or need assistance, don't hesitate to ask for help 😁

### Task 1: Analyze the data

In [ ]:
# To load the dataset
df_train = load_data('train')
df_train.head()

df_validation = load_data('validation')
df_validation.head()

df_test = load_data('test')
df_test.head()

In [ ]:
# Plot some columns of the dataset here

In [ ]:
# Plot some columns of the dataset here

In [ ]:
# Plot some columns of the dataset here

What insigts did you get from the data? What is the effect of the loading conditions on the structure lifetime?

Write your answer here:



...

### Task 2: Train an HSMM

Here are some implemented codes that can train an HSMM

In [ ]:
# Transform the data to the HIMAP format
data_train, max_len = format_data(df_train)

In [ ]:
# Instance the model with some hiper-parameters
n_states = 2
n_iter= 5
model = CustomHSMM(n_states=n_states, # You can change this
                   n_durations = (max_len//n_states)*2, # You can change this
                   n_iter=n_iter, # You can change this
                   name = 'hsmm_test', # You can change this
                   
                   f_value=-0.34, # Please, dont change this.
                   obs_state_len=len(data_train.keys()), # Please, dont change this.
                   )

In [ ]:
# Fit the model parameters
model.fit(data_train)

In [ ]:
# Save the model for later
model.save_model()

In [ ]:
# In case you need to load a trained model (will be useful later)
model = CustomHSMM(name = 'hsmm_test')
model.load_model('hsmm_test')

In [ ]:
alpha = 0.005 # Confidence level used to generate the intervals
# Transform the data to the HIMAP format
data_validation, _ = format_data(df_validation)

print('Mae\t\t\tPINAW\t\t\tPICP\t\t\tCRPS')
for cmd in data_validation.values():
    actual_RUL = len(cmd)-np.arange(len(cmd))
    
    prognostic = predict_hsmm_pdf_staked(cmd, model, [])[0]
    expected, lb, ub = predict_hsmm_bounds(cmd, model,alpha = alpha ) # Function that returns the expected rul and lower and upper values for a confidence intervals
    
    print(mae(expected,actual_RUL), '\t', pinaw(lb,ub),'\t', picp(lb,ub, actual_RUL), '\t', crps(prognostic, actual_RUL))
    
    # Uncomment these lines to get the RUL plot
    plot_rul_bounds(actual_RUL,  expected, lb, ub)
    break
    

    
    

Comment on the reliability of the prognostics. What are the obtained metrics suggesting? What could be improoved?

Write your answer here:



...


### Task 3: Reflect on how could you manage prognostic uncertianty

(hint: What are the sources of uncertainty in this case?)

Write your answer here:



...

### Task 4: Implement your uncertainty management strategy


(hint: use the codes from Task 2)

Now, plot and compare the performance of your initial model with the provided baseline prognostic model and your UM prognostic model

In [ ]:
# Example code to plot multiple curves

# 1. load models 
model1 = CustomHSMM(name = 'hsmm_test')
model1.load_model('hsmm_test')

model2 = CustomHSMM(name = 'hsmm_baseline')
model2.load_model('hsmm_baseline')


alpha = 0.005 # Confidence level used to generate the intervals
for cmd in data_train.values():
    actual_RUL = len(cmd)-np.arange(len(cmd))
    
    predictions = [
        predict_hsmm_bounds(cmd, model1,alpha = alpha ),
        predict_hsmm_bounds(cmd, model2,alpha = alpha ),
                   ] # Agregate predictins in a list
    
    # Uncomment these lines to get the RUL plot
    plot_rul_bounds_multiple(actual_RUL,  predictions)
    break

Comment on the effect of the UM stategy. What did you observe in the metrics? Comment on the reliability of these prognsotics
Write your answer here:



...


### Task 5: Pre compute your prognostics for the training, validation and testing datasets



Compute the predictions with the baseline model called  and the UM model (Task 4)

In [ ]:
# Example on how to generate prognostics with a single model

# 1. Load the model
name = 'hsmm_baseline'
model = CustomHSMM(name = name)
model.load_model(name)

# 2. format the dataset
data_train, _ = format_data(df_train)

# 3.  Genrate the prognostics for each trajectory
predictions_train = []
for trajectory in tqdm(data_train.values()):
    predictions_train = predict_hsmm_pdf_staked(trajectory, model,predictions_train)
    
# 4. Save the prognostics
save_prognostics(predictions_train, name,'train')


In [ ]:
# For validation
# 2. format the dataset
data_val, _ = format_data(df_validation)

# 3.  Genrate the prognostics for each trajectory
predictions_val = []
for trajectory in tqdm(data_val.values()):
    predictions_val = predict_hsmm_pdf_staked(trajectory, model,predictions_val)
    
# 4. Save the prognostics
save_prognostics(predictions_val, name,'validation')

In [ ]:
# For test
# 2. format the dataset
data_test, _ = format_data(df_test)

# 3.  Genrate the prognostics for each trajectory
predictions_test = []
for trajectory in tqdm(data_test.values()):
    predictions_test = predict_hsmm_pdf_staked(trajectory, model,predictions_test)
    
# 4. Save the prognostics
save_prognostics(predictions_test, name,'test')

# Section 2 - Decision making 🔍 📊

## Tasks description

After you have trained a prognostic model, now your task is to define a policy takes achieves a good maintenance performance. The performance measure is the maintenance cost rate, i.e., 

$$\frac{E[C]}{E[T]}$$

In the below cells, there are some example benchmark policies that you should be able to beat. Your policy takes as an input at each time step: 

a) the prognostic information, 

b) the current time, 

c) some policy-specific parameters that are passed as a list. 

The custom parameters in c) should be fixed a-priori and should not contain information about the time to failure of the components. Or, in simple terms, do not cheat.

Once you have defined and evaluated your policy as well as the benchmarks, you can visually compare their performances. Feel free to define additional diagnostics that might be useful to you.

### Benchmark policies

In [ ]:
def do_nothing(prog: ndarray, time: int, pol_args: list) -> ndarray:
    """
    Does nothing every time step -> corrective maintenance policy.

    Args:
        prog (N, D): prognostic information
        time: current time/age of the component
        pol_args: list of policy-specific arguments

    Returns:
        acts (N,): array of 0/1 for each component, where 0==DN & 1==PR
    """
    return np.zeros(prog.shape[0])
    
def prob_thres(prog: ndarray, time: int, pol_args: list) -> ndarray:
    """
    Simple probability threshold based policy.

    Args:
        prog (N, D): prognostic information at a certain timestep
        time: current time/age of the component
        pol_args: list of policy-specific arguments

    Returns:
        acts (N,): array of 0/1 for each component, where 0==DN & 1==PR
    """
    a = np.zeros(len(prog)) 

    ub = pol_args[0]
    pf_thres = pol_args[1]
    
    supp = np.arange(prog.shape[1])

    # first find x of support, then interpolate
    cdf = np.cumsum(prog, axis=1) # (N, D)
    pf_preds = np.array(
        [np.interp(x=ub, xp=supp, fp=cdf[k,:]) for k in range(cdf.shape[0])]
    ).squeeze() # (N,)

    a[pf_preds > pf_thres] = 1
    return a

def age_pol(prog: ndarray, time: int, pol_args: list) -> ndarray:
    """
    Age-based replacement policy: components either fail or are replaced at the
    floored replacement age.

    Args:
        prog (N, D): prognostic information
        time: current time/age of the component
        pol_args: list of policy-specific arguments

    Returns:
        acts (N,): array of 0/1 for each component, where 0==DN & 1==PR
    """
    t_rep = pol_args[0]
    Delta_T = pol_args[1]
    t_rep_floor = (t_rep // Delta_T) * Delta_T

    if time == t_rep_floor:
        a = np.ones(prog.shape[0])
    else:
        a = np.zeros(prog.shape[0]) 
    return a

### Create your own policy

In [ ]:
def mypol(prog: list, time: int, pol_args: Any) -> ndarray:
    """
    General template for the custom policy.

    Args:
        prog (N, D): list of prognostic information for a batch of N components
        time: the current time passed
        pol_args: list of policy-specific arguments
    
    Returns:
        acts (N,): array of 0/1 for each component, where 0==DN & 1==PR
    """
    acts = np.zeros(len(prog))

    ## TODO: populate now entries of action vector based on chosen heuristic

    return acts

### Fine tune your policy with the validation dataset

Compare the performance with the benchmanrk policies

In [ ]:
# create ReplacementAgent object
A = ReplacementAgent(prog_model_name='hsmm_baseline',cf=config)

In [ ]:
# evaluate the benchmark policies
c_opt, t_opt, cr_opt, var_cr_opt = A.opt_pol(tfs=A.trainval_tfs)

c_dn, t_dn, cr_dn, var_cr_dn = A.eval_pol(pol=do_nothing, tfs=A.trainval_tfs, prog=A.trainval_prog)

prob_pol_args = [config["Delta_T"], config["c_p"] / config["c_c"]]
c_prob, t_prob, cr_prob, var_cr_prob = A.eval_pol(pol=prob_thres, tfs=A.trainval_tfs, prog=A.trainval_prog, pol_args=prob_pol_args)

age_pol_args = [A.trainval_tfs.mean(), config["Delta_T"]]
c_age, t_age, cr_age, var_cr_age = A.eval_pol(pol=age_pol, tfs=A.trainval_tfs, prog=A.trainval_prog, pol_args=age_pol_args)

In [ ]:
# TODO: define here the parameters of your policy and evaluate it
my_pol_args = [A.trainval_tfs.mean(), config["Delta_T"]]
c_my, t_my, cr_my, var_cr_my = A.eval_pol(pol=mypol, tfs=A.trainval_tfs, prog=A.trainval_prog, pol_args=age_pol_args)

In [ ]:
results = {}

if "cr_opt" in globals():
    results["opt_pol"] = (c_opt, t_opt, cr_opt, var_cr_opt)

if "cr_dn" in globals():
    results["do_nothing"] = (c_dn, t_dn, cr_dn, var_cr_dn)

if "cr_age" in globals():
    results["age"] = (c_age, t_age, cr_age, var_cr_age)

if "cr_prob" in globals():
    results["prob_thres"] = (c_prob, t_prob, cr_prob, var_cr_prob)

if "cr_my" in globals():
    results["my_policy"] = (c_my, t_my, cr_my, var_cr_my)

vis.plot_policy_comparison(results=results)
vis.plot_cp_vs_cc(results=results)

### Evaluate your policy with the testing dataset

In [ ]:
# evaluate the benchmark policies
c_opt_test, t_opt_test, cr_opt_test, var_cr_opt_test = A.opt_pol(tfs=A.test_tfs)

c_dn_test, t_dn_test, cr_dn_test, var_cr_dn_test = A.eval_pol(pol=do_nothing, tfs=A.test_tfs, prog=A.test_prog)

prob_pol_args = [config["Delta_T"], config["c_p"] / config["c_c"]]
c_prob_test, t_prob_test, cr_prob_test, var_cr_prob_test = A.eval_pol(pol=prob_thres, tfs=A.test_tfs, prog=A.test_prog, pol_args=prob_pol_args)

age_pol_args = [A.test_tfs.mean(), config["Delta_T"]]
c_age_test, t_age_test, cr_age_test, var_cr_age_test = A.eval_pol(pol=age_pol, tfs=A.test_tfs, prog=A.test_prog, pol_args=age_pol_args)

c_my_test, t_my_test, cr_my_test, var_cr_my_test = A.eval_pol(pol=mypol, tfs=A.test_tfs, prog=A.test_prog, pol_args=age_pol_args)

test_results = {}

if "cr_opt" in globals():
    test_results["opt_pol"] = (c_opt_test, t_opt_test, cr_opt_test, var_cr_opt_test)

if "cr_dn" in globals():
    test_results["do_nothing"] = (c_dn_test, t_dn_test, cr_dn_test, var_cr_dn_test)

if "cr_age" in globals():
    test_results["age"] = (c_age_test, t_age_test, cr_age_test, var_cr_age_test)

if "cr_prob" in globals():
    test_results["prob_thres"] = (c_prob_test, t_prob_test, cr_prob_test, var_cr_prob_test)

if "cr_my" in globals():
    test_results["my_policy"] = (c_my_test, t_my_test, cr_my_test, var_cr_my_test)

vis.plot_policy_comparison(results=test_results)
vis.plot_cp_vs_cc(results=test_results)

# Bonus:

Use your porgnostic model with uncertainty management, as well as your optimized policy. 

Compare your performance with others

In [ ]:
# Load your Uncertainty management model
model_name = ''
fold = 'test'

model = CustomHSMM(name = model_name)
model.load_model(model_name)

# 2. format the dataset
df = load_data(fold)
data_formated, _ = format_data(df)

predictions = []
for trajectory in tqdm(data_formated.values()):
    predictions = predict_hsmm_pdf_staked(trajectory, model,predictions)

In [ ]:

# create ReplacementAgent object
A = ReplacementAgent(prog_model_name=model_name,cf=config)


tfs, prog = A.process_prognostics(predictions)

In [ ]:
# evaluate the benchmark policies
c_opt_test, t_opt_test, cr_opt_test, var_cr_opt_test = A.opt_pol(tfs=tfs)

c_dn_test, t_dn_test, cr_dn_test, var_cr_dn_test = A.eval_pol(pol=do_nothing, tfs=tfs, prog=prog)

prob_pol_args = [config["Delta_T"], config["c_p"] / config["c_c"]]
c_prob_test, t_prob_test, cr_prob_test, var_cr_prob_test = A.eval_pol(pol=prob_thres, tfs=tfs, prog=prog, pol_args=prob_pol_args)

age_pol_args = [tfs.mean(), config["Delta_T"]]
c_age_test, t_age_test, cr_age_test, var_cr_age_test = A.eval_pol(pol=age_pol, tfs=tfs, prog=prog, pol_args=age_pol_args)

c_my_test, t_my_test, cr_my_test, var_cr_my_test = A.eval_pol(pol=mypol, tfs=tfs, prog=prog, pol_args=age_pol_args)

test_results = {}

if "cr_opt_test" in globals():
    test_results["opt_pol"] = (c_opt_test, t_opt_test, cr_opt_test, var_cr_opt_test)

if "cr_dn_test" in globals():
    test_results["do_nothing"] = (c_dn_test, t_dn_test, cr_dn_test, var_cr_dn_test)

if "cr_age_test" in globals():
    test_results["age"] = (c_age_test, t_age_test, cr_age_test, var_cr_age_test)

if "cr_prob_test" in globals():
    test_results["prob_thres"] = (c_prob_test, t_prob_test, cr_prob_test, var_cr_prob_test)

if "cr_my_test" in globals():
    test_results["my_policy"] = (c_my_test, t_my_test, cr_my_test, var_cr_my_test)

vis.plot_policy_comparison(results=test_results)
vis.plot_cp_vs_cc(results=test_results)